# Basics: Data Loading & GPS Visualization

This notebook demonstrates the fundamentals of loading and visualizing AIM™ telemetry data using libxrk.

## What You'll Find Here

- **Data Loading**: Load `.xrk` or `.xrz` files without AIM software
- **Lap Times Table**: View all recorded laps with computed lap times
- **GPS Speed Map**: Visualize speed around the track on an interactive map
- **Brake & Throttle Overlay**: See driver inputs overlaid on the track map

## Using Your Own Data

To analyze your own data:

1. **Run the file picker cell** below to display the upload widget
2. **Drag and drop** your `.xrk` or `.xrz` file onto the upload button (or click to browse)
3. **Run all remaining cells** to analyze your data

The status indicator will show which file is being used. If you don't upload a file, the sample data will be used.

## Requirements

- GPS data channels (`GPS Latitude`, `GPS Longitude`, `GPS Speed`)
- Brake pressure (`BrakePress`) and throttle (`PPS`) for the inputs overlay

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [ ]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q pandas plotly libxrk motorsports-data-notebook jinja2 ipywidgets

In [ ]:
# Import core libraries
import pandas as pd

# Visualization libraries
import plotly.graph_objects as go

# Import libxrk:
from libxrk import aim_xrk

# Import helper functions (includes show_fig for JupyterLite compatibility)
from motorsports_data_notebook import show_fig, get_best_lap, plot_lap_gps, FileUpload

In [ ]:
# File picker - upload your own .xrk/.xrz file or use the sample data
file_upload = FileUpload(default_file="CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz")
file_upload.display()

In [ ]:
# Load the data file
log = aim_xrk(file_upload.get_file_data())

In [ ]:
# Flatten all the data into a uniform table, interpolating as needed (can use a lot of RAM!)
channels = log.get_channels_as_table().to_pandas()

# Add derived columns
channels["speed_kmh"] = channels["GPS Speed"] * 3.6

In [ ]:
# Load laps and compute lap times
laps = log.laps.to_pandas()
laps["lap_time"] = pd.to_timedelta(laps["end_time"] - laps["start_time"], unit="ms")

laps.style.format(
    {"lap_time": lambda x: f"{int(x.total_seconds() // 60)}:{x.total_seconds() % 60:06.3f}"}
)

In [ ]:
# Best lap extraction
best_lap = get_best_lap(laps)
start_ts = best_lap["start_time"]
end_ts = best_lap["end_time"]
# Use < for end_ts to exclude the first sample of the next lap (where distance resets to 0)
lap_channels = channels.query(f"timecodes >= @start_ts and timecodes < @end_ts").copy()

In [ ]:
# plot speed on GPS map
fig = plot_lap_gps(
    lat=lap_channels["GPS Latitude"],
    lon=lap_channels["GPS Longitude"],
    color_channels=[(lap_channels["speed_kmh"], "Speed (km/h)", "Viridis")],
    title="Speed",
)
show_fig(fig)

In [ ]:
# Plot with multiple color channels
fig = plot_lap_gps(
    lat=lap_channels["GPS Latitude"],
    lon=lap_channels["GPS Longitude"],
    color_channels=[
        (lap_channels["BrakePress"], "BrakePres", "Reds"),
        (lap_channels["PPS"], "Throttle", "Greens"),
    ],
    title="Accelerator and Brake Pressure on Best Lap",
)
show_fig(fig)